In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
try:
    from Preprocessing.analysis_utils import get_stimulus_timestamps, get_running_timestamps, get_running_df, resample_dff_for_trial, resample_running_for_trial
except ModuleNotFoundError:
    from analysis_utils import get_stimulus_timestamps, get_running_timestamps, get_running_df, resample_dff_for_trial, resample_running_for_trial
from hdmf_zarr.nwb import NWBZarrIO
import matplotlib.pyplot as plt
import anndata as ad
import pickle
from scipy.interpolate import interp1d
from scipy.optimize import curve_fit
from pathlib import Path
import os
import sys
import time
import zarr
import numcodecs
import shutil
import re
RESAMPLE_RATE = 10    # Hz
PRE_STIM      = 1.0   # seconds before stimulus onset
POST_STIM     = 1.0   # seconds after stimulus offset

SCRATCH_DIR = Path("/root/capsule/scratch")

DATA_DIR = Path("/root/capsule/data")
STIM_ALIGNED_DIR = SCRATCH_DIR / "ophys" / "stim_aligned"
MORPH_DIR = SCRATCH_DIR / "morphology"
TRANSCRIPT_DIR = SCRATCH_DIR / 'transcriptomics'


MULTIMODAL = SCRATCH_DIR / "multimodal_data"
MULTIMODAL.mkdir(parents=True, exist_ok=True)

# Import tuning and GLM functions from code/functions/
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('.'), '..')))
from functions.tuning import compute_tuning_for_session, save_tuning_to_zarr
from functions.glm import add_glm_aggregate_columns

In [ ]:
data_info = pickle.load(open('/root/capsule/code/Preprocessing/data_info.pkl', 'rb'))
mouse_ids = list(data_info.keys())
print('mouse_ids:', mouse_ids)

- adata.X ← avergae df/f during 2s of stimuli
- adata.layers["baseline"] ← average df/f during 1s gray screenbefore the sitmuli
- adata.obs ← cell metadata
- adata.var ← trial metadata

In [ ]:
stim_type = 'GratingStim'
running_thr = 1.0
for mouse_id in mouse_ids:
    X = []
    UID = []
    VAR = []
    for session_name in ['session_1', 'session_2', 'session_3']:
        in_path = STIM_ALIGNED / f"{mouse_id}_{session_name}_coregistered.pkl"
        with open(in_path, 'rb') as f:
            session_data = pickle.load(f)

        var = session_data[stim_type]['trial_info']
        var = var[~var['gray_screen']].reset_index(drop=True)
        target_slice = session_data[stim_type]['running'][1::2,10:-10,0]
        if np.count_nonzero(~np.isnan(target_slice)) == 0:
            avg_running = np.nan
        else:
            avg_running = np.nanmean(target_slice, axis=1)
        print(session_data[stim_type]['running'].shape)
        var['avg_running'] = avg_running
        var['is_running'] = avg_running>running_thr
        var['day'] = int(session_name.split('_')[-1])
        var['date'] = data_info[mouse_id]['ophys'][session_name]['raw'].split('_')[-2]
        VAR.append(var)
        x1 = np.nanmean(session_data[stim_type]['dff']['data'][1::2,10:-10], axis=1)
        x2 = np.nanmean(session_data[stim_type]['dff']['data'][::2,10:-10], axis=1)
        X.append(np.stack([x1, x2]))
        UID.append(session_data[stim_type]['dff']['unique_ids'])

    VAR = pd.concat(VAR, axis=0).reset_index(drop=True)
    X = np.concatenate(X, axis=1)
    assert all(np.array_equal(UID[0], u) for u in UID[1:]), "UID vectors are not equal across sessions"
    print("✓ All UID vectors are equal")
    UID = UID[0]
    OBS = cell_types_coreg_df.merge(cellxgene_coreg_df, on=[col for col in cell_types_coreg_df.columns if col in cellxgene_coreg_df.columns]).set_index('unique_id',drop=True).loc[UID].reset_index()
    adata = ad.AnnData(
        X=X[0].T,
        obs=OBS.reset_index(drop=True),
        var=VAR.reset_index(drop=True),
        layers={"baseline": X[1].T},
    )

    out_path = STIM_ALIGNED / f"{mouse_id}_GratingStim.h5ad"
    adata.write_h5ad(out_path)
    print(adata)
    print(f"Saved to {out_path}")

FileNotFoundError: [Errno 2] No such file or directory: '/root/capsule/scratch/stim_aligned_dff/778174_session_1_coregistered.pkl'